# SupplyForge AI 2.0: KAN-28 - SKU Metadata Mapping



Tài liệu này thực hiện ánh xạ các thuộc tính sản phẩm tĩnh (`brand`, `categoryId`, `price`, `cost`) từ danh mục sản phẩm sạch `products_cleaned.json` vào lưới chuỗi thời gian liên tục `calendar_daily_sales.csv`.



Mỗi quan sát hàng ngày (date x itemId) sẽ được bổ sung các đặc trưng tĩnh này, phục vụ trực tiếp cho quá trình Feature Engineering (Target Encoding, Frequency Encoding) và huấn luyện mô hình dự báo.


### Quy tắc nghiệp vụ và Tiêu chuẩn Nghiệm thu (Acceptance Criteria):

1. **Loại bỏ trùng lặp**: Lọc sạch trùng lặp `itemId` trong danh mục sản phẩm trước khi join để tránh làm tăng số dòng của lưới.

2. **Left Join**: Thực hiện ghép lưới thời gian với danh mục sản phẩm theo cột khóa chung `itemId`.

3. **Điền khuyết (Imputation)**: Đối với các active SKU trong lưới chuỗi thời gian không khớp với danh mục sản phẩm (Unmatched SKUs), điền khuyết thuộc tính tĩnh bằng:

   - `brand` -> `'unknown_brand'`

   - `categoryId` -> `'unknown_category'`

   - `price` -> Trung vị giá bán toàn cầu (`250,000.00`)

   - `cost` -> Trung vị giá vốn toàn cầu (`18,500.00`)

4. **Bảo toàn số dòng**: Đảm bảo số lượng dòng của lưới thời gian trước và sau khi merge trùng khớp hoàn toàn.

5. **Đầu ra**: Xuất file dữ liệu tích hợp `calendar_daily_sales_metadata.csv`.


In [1]:
import pandas as pd

import numpy as np

import json

import os



DATA_DIR = '/home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH'

grid_path = os.path.join(DATA_DIR, 'calendar_daily_sales.csv')

prod_path = os.path.join(DATA_DIR, 'products_cleaned.json')



print("Đang tải dữ liệu...")

df_grid = pd.read_csv(grid_path, low_memory=False)

df_prod = pd.read_json(prod_path)



print(f"Tổng số dòng trong lưới Daily: {len(df_grid)}")

print(f"Tổng số sản phẩm trong danh mục: {len(df_prod)}")


Đang tải dữ liệu...


Tổng số dòng trong lưới Daily: 323250
Tổng số sản phẩm trong danh mục: 1335


In [2]:
print("Đang tiền xử lý danh mục sản phẩm...")



# 1. Kiểm tra và loại bỏ trùng lặp itemId

dup_count = df_prod['itemId'].duplicated().sum()

print(f"Số lượng itemId bị trùng lặp ban đầu: {dup_count}")

df_prod_unique = df_prod.drop_duplicates(subset=['itemId']).copy()

print(f"Số sản phẩm sau khi lọc trùng lặp itemId: {len(df_prod_unique)}")



# 2. Đồng nhất kiểu dữ liệu sang chuỗi

df_grid['itemId'] = df_grid['itemId'].astype(str)

df_prod_unique['itemId'] = df_prod_unique['itemId'].astype(str)


Đang tiền xử lý danh mục sản phẩm...
Số lượng itemId bị trùng lặp ban đầu: 0
Số sản phẩm sau khi lọc trùng lặp itemId: 1335


In [3]:
print("Đang thực hiện Left Join để ánh xạ thuộc tính sản phẩm...")

meta_cols = ['itemId', 'brand', 'categoryId', 'price', 'cost']

df_merged = pd.merge(df_grid, df_prod_unique[meta_cols], on='itemId', how='left')

print(f"Tổng số dòng sau khi ghép nối: {len(df_merged)}")


Đang thực hiện Left Join để ánh xạ thuộc tính sản phẩm...
Tổng số dòng sau khi ghép nối: 323250


In [4]:
print("=== PHÂN TÍCH SKUs CHƯA KHỚP NỐI ===")

unmatched_skus = df_merged[df_merged['brand'].isnull()]['itemId'].unique()

print(f"Số lượng active SKUs không tìm thấy trong danh mục sản phẩm: {len(unmatched_skus)}")

print("Mẫu các SKUs chưa khớp nối:", unmatched_skus[:10])



print("\nTỷ lệ khuyết thiếu ban đầu:")

for col in ['brand', 'categoryId', 'price', 'cost']:

    missing_pct = df_merged[col].isnull().mean() * 100

    print(f" - {col}: {missing_pct:.4f}%")


=== PHÂN TÍCH SKUs CHƯA KHỚP NỐI ===
Số lượng active SKUs không tìm thấy trong danh mục sản phẩm: 64
Mẫu các SKUs chưa khớp nối: <StringArray>
[ '9442469665',  '4685444679',  '9547367295',  '3385539129',  '9542108115',
  '5085870493',  '3049042271',  '6985868323', '11950524512', '10725446366']
Length: 10, dtype: str

Tỷ lệ khuyết thiếu ban đầu:
 - brand: 17.0667%
 - categoryId: 60.2667%
 - price: 7.2000%
 - cost: 7.2000%


In [5]:
print("Đang tiến hành điền khuyết thuộc tính tĩnh theo quy tắc nghiệp vụ...")



# 1. Tính toán giá trị trung vị toàn cầu làm fallback

global_median_price = df_prod_unique.loc[df_prod_unique['price'] > 0, 'price'].median()

global_median_cost = df_prod_unique.loc[df_prod_unique['cost'] > 0, 'cost'].median()



if pd.isna(global_median_price) or global_median_price <= 0:

    global_median_price = 250000.0

if pd.isna(global_median_cost) or global_median_cost <= 0:

    global_median_cost = 18500.0



print(f"Trung vị giá bán toàn cầu: {global_median_price:,.2f}đ")

print(f"Trung vị giá vốn toàn cầu: {global_median_cost:,.2f}đ")



# 2. Điền khuyết các giá trị Null

df_merged['brand'] = df_merged['brand'].fillna('unknown_brand')

df_merged['categoryId'] = df_merged['categoryId'].fillna('unknown_category')

df_merged['price'] = df_merged['price'].fillna(global_median_price)

df_merged['cost'] = df_merged['cost'].fillna(global_median_cost)



print("\nTỷ lệ khuyết thiếu sau khi điền:")

for col in ['brand', 'categoryId', 'price', 'cost']:

    missing_pct = df_merged[col].isnull().mean() * 100

    print(f" - {col}: {missing_pct:.4f}%")


Đang tiến hành điền khuyết thuộc tính tĩnh theo quy tắc nghiệp vụ...
Trung vị giá bán toàn cầu: 280,000.00đ
Trung vị giá vốn toàn cầu: 15,500.00đ

Tỷ lệ khuyết thiếu sau khi điền:
 - brand: 0.0000%
 - categoryId: 0.0000%
 - price: 0.0000%
 - cost: 0.0000%


In [6]:
print("=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===\n")



# 1. Số lượng dòng không đổi

assert len(df_merged) == len(df_grid), f"Lỗi: Số dòng thay đổi từ {len(df_grid)} sang {len(df_merged)}"

print(f"1. Số lượng dòng trước và sau khi merge nhất quán: Đạt ({len(df_merged)} dòng)")



# 2. Không trùng lặp itemId trong products master

assert df_prod_unique['itemId'].duplicated().sum() == 0, "Lỗi: Có trùng lặp itemId trong bảng sản phẩm gốc"

print("2. Không có trùng lặp itemId trong danh mục sản phẩm: Đạt")



# 3. Tỷ lệ khuyết thiếu của các trường quan trọng là 0%

for col in ['brand', 'categoryId', 'price', 'cost']:

    missing_cnt = df_merged[col].isnull().sum()

    assert missing_cnt == 0, f"Lỗi: Phát hiện {missing_cnt} giá trị khuyết ở cột {col}"

print("3. Tỷ lệ khuyết thiếu của brand, categoryId, price, cost là 0%: Đạt")



# 4. Mọi SKU trong lưới đều có metadata hợp lệ

active_grid_skus = df_merged['itemId'].unique()

unmatched_in_merged = df_merged[df_merged['brand'] == 'unknown_brand']['itemId'].unique()

print(f"4. Tổng số SKU trong lưới: {len(active_grid_skus)}")

print(f"   Số SKU được điền giá trị mặc định (unmatched): {len(unmatched_in_merged)}")

print("   Tất cả active SKUs đã nhận thành công metadata: Đạt")


=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===

1. Số lượng dòng trước và sau khi merge nhất quán: Đạt (323250 dòng)
2. Không có trùng lặp itemId trong danh mục sản phẩm: Đạt
3. Tỷ lệ khuyết thiếu của brand, categoryId, price, cost là 0%: Đạt


4. Tổng số SKU trong lưới: 375
   Số SKU được điền giá trị mặc định (unmatched): 64
   Tất cả active SKUs đã nhận thành công metadata: Đạt


In [7]:
print("=== LƯU TRỮ DỮ LIỆU ĐÃ ÁNH XẠ METADATA ===")

output_path = os.path.join(DATA_DIR, 'calendar_daily_sales_metadata.csv')

df_merged.to_csv(output_path, index=False)

print(f"Đã lưu kết quả thành công tại: {output_path}")


=== LƯU TRỮ DỮ LIỆU ĐÃ ÁNH XẠ METADATA ===


Đã lưu kết quả thành công tại: /home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH/calendar_daily_sales_metadata.csv
